# Day 4 — Capstone: Semantic Search Engine

---

Time to build the real thing. Today we glue together everything from the last 5 days into a **Semantic Search API** that ingests real PDFs, indexes them into ChromaDB, and answers search queries over FastAPI.

## What you'll build

A FastAPI app exposing:

- `POST /ingest` — accepts a PDF path, extracts text, chunks it, embeds, stores in Chroma
- `GET /search?q=...&top_k=5&source=...` — semantic search with optional metadata filter
- `GET /stats` — how many chunks are indexed, from which sources

Everything runs locally with **no API keys**. Ties directly back to Section 2 (FastAPI).


## Architecture in one picture

```
     PDF file
        │
        ▼
   ┌─────────┐
   │  pypdf  │  extract text page by page
   └─────────┘
        │
        ▼
   ┌────────────────────┐
   │  recursive_chunk   │  ~500 chars, 50 overlap
   └────────────────────┘
        │  (list of chunks + page numbers)
        ▼
   ┌────────────────────┐
   │ sentence-trans.    │  MiniLM → 384-dim vectors
   └────────────────────┘
        │
        ▼
   ┌────────────────────┐
   │      ChromaDB      │  persistent, on disk
   └────────────────────┘
        │
        ▼
   ┌────────────────────┐
   │      FastAPI       │  /ingest  /search  /stats
   └────────────────────┘
```

Every arrow is a function you already know how to write.


## 1. Setup — install everything


In [ ]:
!pip install fastapi uvicorn chromadb sentence-transformers pypdf --quiet

## 2. The building blocks — reuse from previous days

We're going to reuse **exactly** what we built:
- `recursive_chunk()` from Day 3
- ChromaDB persistent client from Day 2
- Sentence Transformers embedding from Day 1.1
- Metadata filtering from Day 2

The only new piece is **PDF extraction** using `pypdf`.


In [ ]:
from pypdf import PdfReader

def extract_pages(pdf_path: str) -> list[dict]:
    """Return [{'text': ..., 'page': 1}, ...] for every page."""
    reader = PdfReader(pdf_path)
    pages = []
    for i, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        if text.strip():
            pages.append({"text": text, "page": i})
    return pages

# Try it on any PDF you have handy
# pages = extract_pages("your_file.pdf")
# print(f"Got {len(pages)} pages")
# print(pages[0]["text"][:300])


## 3. The full pipeline

Below is the *whole* app in one cell so you can see the shape. In the real project we split it into a proper `main.py`.


In [ ]:
from pathlib import Path
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import chromadb
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader

# --- Setup (runs once at startup) ---
app = FastAPI(title="Semantic Search API")
model = SentenceTransformer("all-MiniLM-L6-v2")
chroma = chromadb.PersistentClient(path="./search_db")
collection = chroma.get_or_create_collection(name="documents")


def recursive_chunk(text: str, chunk_size: int = 500, overlap: int = 50):
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks, current = [], ""
    for para in paragraphs:
        if len(current) + len(para) + 1 <= chunk_size:
            current = (current + "\n\n" + para).strip()
        else:
            if current:
                chunks.append(current)
            while len(para) > chunk_size:
                chunks.append(para[:chunk_size])
                para = para[chunk_size - overlap:]
            current = para
    if current:
        chunks.append(current)
    return chunks


# --- Schemas ---
class IngestRequest(BaseModel):
    pdf_path: str


# --- Endpoints ---
@app.post("/ingest")
def ingest(req: IngestRequest):
    path = Path(req.pdf_path)
    if not path.exists():
        raise HTTPException(404, f"File not found: {req.pdf_path}")

    reader = PdfReader(str(path))
    all_chunks, all_meta, all_ids = [], [], []
    counter = 0
    for page_num, page in enumerate(reader.pages, start=1):
        text = (page.extract_text() or "").strip()
        if not text:
            continue
        for chunk in recursive_chunk(text, 500, 50):
            all_chunks.append(chunk)
            all_meta.append({"source": path.name, "page": page_num})
            all_ids.append(f"{path.stem}_{counter}")
            counter += 1

    if not all_chunks:
        return {"chunks_added": 0}

    vectors = model.encode(all_chunks).tolist()
    collection.add(documents=all_chunks, embeddings=vectors,
                   metadatas=all_meta, ids=all_ids)
    return {"chunks_added": len(all_chunks), "source": path.name}


@app.get("/search")
def search(q: str, top_k: int = 5, source: str | None = None):
    q_vec = model.encode([q]).tolist()
    where = {"source": source} if source else None
    r = collection.query(query_embeddings=q_vec, n_results=top_k, where=where)
    return {
        "query": q,
        "results": [
            {"text": doc, "source": meta["source"], "page": meta["page"],
             "distance": round(dist, 3)}
            for doc, meta, dist in zip(r["documents"][0], r["metadatas"][0], r["distances"][0])
        ],
    }


@app.get("/stats")
def stats():
    return {"total_chunks": collection.count()}


## 4. Run it

Save the code as `main.py` (see the file in this folder) and start the server:

```bash
uvicorn main:app --reload
```

Then try it out (in another terminal):

```bash
# Ingest a PDF
curl -X POST http://localhost:8000/ingest \
     -H "Content-Type: application/json" \
     -d '{"pdf_path": "/path/to/your/file.pdf"}'

# Search
curl "http://localhost:8000/search?q=what+is+machine+learning&top_k=3"

# Search restricted to one source
curl "http://localhost:8000/search?q=payment&source=contract.pdf"

# Stats
curl "http://localhost:8000/stats"
```

Or, better: open **http://localhost:8000/docs** — FastAPI's built-in Swagger UI lets you try every endpoint from the browser.


## 5. Sanity check the results

Try queries that:

1. **Match keywords** in the PDF (should be perfect)
2. **Don't share any words** with the PDF but ask about the same topic (semantic search should shine)
3. **Ask something not in the PDF** (results should have high distance — they'll still return, but low quality)

That last case is important — semantic search **always returns something**, even if the answer isn't there. Filtering out low-quality hits (distance > some threshold) is a real production concern.


## 6. What to try next (optional)

Once your basic API works, try one of these upgrades:

- **Multiple PDFs** — ingest 3–5 documents. Confirm the `source` filter works.
- **Hybrid search** — add BM25 alongside semantic (Day 3) and blend the scores.
- **A tiny frontend** — one HTML page with a search box calling your API.
- **Distance threshold** — refuse to return results with `distance > 1.0` and reply `"No relevant results found."`

Any of these is a great addition to your portfolio repo.


## Recap — what you've built in Section 5

Over 6 days you went from "what's an embedding?" to a working **semantic search API** with:

- Real PDF ingestion
- Chunking with metadata
- A persistent vector database
- HTTP search endpoint with metadata filtering
- Runs entirely on your laptop, no paid APIs

This is the exact foundation for **Section 6 — RAG**, where you'll take these search results and feed them to an LLM to generate answers with citations.

Great work.
